# Anomaly Detection Model

This notebook trains a simple statistical baseline for IoT anomaly detection using temperature (`temp_c`) and vibration (`vib`) signals. It loads historical data from CSV, keeps the most recent 4 days as the reference period, computes rolling averages, and saves fixed residual thresholds for later inference.

## 1) Initialize Spark

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Anomaly Detection for IoT")
    .enableHiveSupport()
    .getOrCreate())

## 2) Load and prepare the dataset

In [ ]:
from pyspark.sql import functions as F

DATA_PATH = "/home/osbdet/notebooks/Group Project IOT/output.csv"

data = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(DATA_PATH)
    .distinct()
    .withColumn("event_time", F.to_timestamp("event_time"))
    .orderBy("event_time"))

print(f"Rows: {data.count()}")
print(f"Columns: {len(data.columns)}")
data.printSchema()
data.show(5, truncate=False)

## 3) Quick exploratory analysis

Convert only the columns needed for visualization to Pandas.

In [ ]:
import matplotlib.pyplot as plt

pdf = data.select("event_time", "temp_c", "vib").toPandas().sort_values("event_time")

plt.figure()
plt.hist(pdf["temp_c"], bins=30)
plt.title("Histogram of Temperature (temp_c)")
plt.xlabel("Temperature (°C)")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.hist(pdf["vib"], bins=30)
plt.title("Histogram of Vibration (vib)")
plt.xlabel("Vibration")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.plot(pdf["event_time"], pdf["temp_c"], label="Temperature (°C)")
plt.plot(pdf["event_time"], pdf["vib"], label="Vibration")
plt.title("Temperature and Vibration Over Time")
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

pdf["time_diff"] = pdf["event_time"].diff()
print(pdf["time_diff"].describe())

## 4) Keep the recent reference window

The model is fitted on the most recent 4 days of data. This keeps the baseline closer to current device behavior while still using a fixed, offline-trained setup.

In [ ]:
from datetime import timedelta

latest_time = data.select(F.max("event_time").alias("latest_time")).collect()[0]["latest_time"]
cutoff_time = latest_time - timedelta(days=4)

filtered_data = data.filter(F.col("event_time") >= cutoff_time)

print("Latest timestamp:", latest_time)
print("Cutoff timestamp:", cutoff_time)
print("Rows after filtering:", filtered_data.count())

filtered_data.selectExpr(
    "min(event_time) as min_time",
    "max(event_time) as max_time"
).show()

## 5) Inspect the filtered training set

In [ ]:
pdf_recent = filtered_data.select("event_time", "temp_c", "vib").toPandas().sort_values("event_time")

plt.figure()
plt.hist(pdf_recent["temp_c"], bins=30)
plt.title("Filtered Temperature Distribution")
plt.xlabel("Temperature (°C)")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.hist(pdf_recent["vib"], bins=30)
plt.title("Filtered Vibration Distribution")
plt.xlabel("Vibration")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.plot(pdf_recent["event_time"], pdf_recent["temp_c"], label="Temperature (°C)")
plt.plot(pdf_recent["event_time"], pdf_recent["vib"], label="Vibration")
plt.title("Filtered Signals Over Time")
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6) Train the baseline anomaly model

This model uses rolling means as the expected signal and residual standard deviation as the fixed anomaly scale. It stores only the learned configuration for later use in streaming/inference code.

In [ ]:
from pyspark.sql.window import Window
import json

WINDOW_SIZE = 30
K = 3.0
OUTPUT_PATH = "/home/osbdet/notebooks/Group Project IOT/anomaly_model_config.json"

df = (filtered_data
    .withColumn("temp_c", F.col("temp_c").cast("double"))
    .withColumn("vib", F.col("vib").cast("double"))
    .dropna(subset=["event_time", "temp_c", "vib"]))

window_spec = Window.orderBy("event_time").rowsBetween(-WINDOW_SIZE + 1, 0)

df_features = (df
    .withColumn("temp_roll_mean", F.avg("temp_c").over(window_spec))
    .withColumn("vib_roll_mean", F.avg("vib").over(window_spec))
    .withColumn("temp_resid", F.col("temp_c") - F.col("temp_roll_mean"))
    .withColumn("vib_resid", F.col("vib") - F.col("vib_roll_mean"))
    .filter(F.col("temp_roll_mean").isNotNull()))

stats = df_features.agg(
    F.stddev_samp("temp_resid").alias("temp_resid_std"),
    F.stddev_samp("vib_resid").alias("vib_resid_std")
).collect()[0]

model_config = {
    "window_size": WINDOW_SIZE,
    "k_multiplier": K,
    "temp_resid_std": float(stats["temp_resid_std"]),
    "vib_resid_std": float(stats["vib_resid_std"])
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(model_config, f, indent=2)

print("Saved model config:")
print(json.dumps(model_config, indent=2))

## 7) Notes

- This is an offline, fixed-threshold baseline model.
- It does **not** retrain continuously unless this notebook is run again.
- The exported JSON can be used by a separate streaming detector to score incoming sensor events.